In [1]:
# appling what we did on mutli human two players at the same time
# using YOLO for decteing two players at the same time , ball , and racket
!pip install -q mediapipe opencv-python-headless ultralytics

import os, cv2, warnings
import numpy as np
import mediapipe as mp
import tensorflow as tf
from ultralytics import YOLO
from google.colab import files
warnings.filterwarnings('ignore')

print("mediapipe version:", mp.__version__)
print(" All imports ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
mediapipe version: 0.10.33
 All imports ready


In [3]:

print("\n1. Upload tennis_cnn_lstm_v1.h5")
u1 = files.upload()
camera_model = tf.keras.models.load_model(list(u1.keys())[0])
print(f" Camera CNN-LSTM — input: {camera_model.input_shape}")


print("\n2. Loading YOLOv8")
yolo = YOLO('yolov8n.pt')
print(" YOLO loaded")


print("\n3. Upload Video 4 2player ")
uv4 = files.upload()
V4_VIDEO = list(uv4.keys())[0]
print(f" {V4_VIDEO}")


1. Upload tennis_cnn_lstm_v1.h5


Saving tennis_cnn_lstm_v1.h5 to tennis_cnn_lstm_v1 (1).h5
 Camera CNN-LSTM — input: (None, 30, 22)

2. Loading YOLOv8
 YOLO loaded

3. Upload Video 4 2player 


Saving 2Player.mp4 to 2Player (1).mp4
 2Player (1).mp4


In [7]:

import urllib.request, os
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision


SELECTED_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27]
WINDOW_SIZE     = 30
THRESHOLD       = 0.82
STRIDE          = 5
COOLDOWN_FRAMES = 180

LABEL_MAP    = {0: 'Forehand', 1: 'Backhand', 2: 'Serve'}


PLAYER_COLORS = {
    'P1': (0, 200, 80),
    'P2': (200, 80,  0),
}
STROKE_COLORS = {
    0: (0,   200,  80),
    1: (0,    80, 200),
    2: (200,   0,  80),
}
print(" Constants defined")


MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
MODEL_PATH = "pose_landmarker.task"
if not os.path.exists(MODEL_PATH):
    print("Downloading pose landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Downloaded")
else:
    print("Model heree")

base_opts = mp_python.BaseOptions(model_asset_path=MODEL_PATH)

def make_pose_detector():
    opts = mp_vision.PoseLandmarkerOptions(
        base_options=base_opts,
        running_mode=mp_vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.4,
        min_pose_presence_confidence=0.4,
        min_tracking_confidence=0.5
    )
    return mp_vision.PoseLandmarker.create_from_options(opts)


def detect_ball_hsv(frame):
    """HSV ball detection on full fram"""
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([25,120,140]), np.array([40,255,255]))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best, best_score = None, 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 40 or area > 600: continue
        perim = cv2.arcLength(cnt, True)
        if perim == 0: continue
        circ  = 4 * np.pi * area / (perim**2)
        score = area * circ
        if score > best_score and circ > 0.65:
            best_score = score
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                best = (int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"]))
    return best

def is_ball_near_player(ball_pos, skel_frame, full_width, full_height):
    if ball_pos is None: return False
    bx = ball_pos[0] / full_width
    by = ball_pos[1] / full_height
    for idx in range(6):
        jx, jy = skel_frame[idx]
        if np.sqrt((bx - jx)**2 + (by - jy)**2) < 0.25:
            return True
    return False

def extract_skeleton_from_crop(pose_detector, frame, box):

    x1, y1, x2, y2 = box
    h_frame, w_frame = frame.shape[:2]
    pad = 20
    x1p = max(0, x1 - pad)
    y1p = max(0, y1 - pad)
    x2p = min(w_frame, x2 + pad)
    y2p = min(h_frame, y2 + pad)
    crop = frame[y1p:y2p, x1p:x2p]
    if crop.size == 0:
        return None
    crop_w = x2p - x1p
    crop_h = y2p - y1p
    rgb_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_crop)
    result = pose_detector.detect(mp_image)
    if not result.pose_landmarks or len(result.pose_landmarks) == 0:
        return None
    lm = result.pose_landmarks[0]
    pts = []
    for i in SELECTED_JOINTS:
        fx = (lm[i].x * crop_w + x1p) / w_frame
        fy = (lm[i].y * crop_h + y1p) / h_frame
        pts.append([fx, fy])
    return pts

def hip_normalise_and_predict(buffer):

    seq    = np.array(buffer)
    hip_x  = seq[:, 12]
    hip_y  = seq[:, 13]
    seq_3d = seq.reshape(WINDOW_SIZE, 11, 2)
    hip_3d = np.stack([hip_x, hip_y], axis=1)[:, None, :]
    seq_3d = seq_3d - hip_3d
    seq_inp = seq_3d.reshape(1, WINDOW_SIZE, 22)
    probs = camera_model.predict(seq_inp, verbose=0)[0]
    cls   = int(np.argmax(probs))
    conf  = float(probs[cls])
    return cls, conf

print(" Helper functions defined")

 Constants defined
Model heree
 Helper functions defined


In [8]:


SELECTED_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27]
WINDOW_SIZE     = 30
THRESHOLD       = 0.70
STRIDE          = 5
COOLDOWN_FRAMES = 180

LABELS        = ['Forehand', 'Backhand', 'Serve']
LABEL_MAP     = {0: 'Forehand', 1: 'Backhand', 2: 'Serve'}
PLAYER_COLORS = {'P1': (0,200,80), 'P2': (200,80,0)}
STROKE_COLORS = {0: (0,200,80), 1: (0,80,200), 2: (200,0,80)}


CLS_PERSON = 0
CLS_BALL   = 32
CLS_RACKET = 38

pose_p1 = make_pose_detector()
pose_p2 = make_pose_detector()


cap    = cv2.VideoCapture(V4_VIDEO)
fps    = cap.get(cv2.CAP_PROP_FPS) or 60.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_name = V4_VIDEO.replace('.mp4','') + '_multi_human.mp4'
out = cv2.VideoWriter(output_name,
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps, (width, height))

print(f" Processing: {fps:.0f}fps  {width}x{height}  {total} frames")
print(f"   Output -> {output_name}")
print("-" * 55)


buffers          = {'P1': [], 'P2': []}
last_det         = {'P1': -COOLDOWN_FRAMES, 'P2': -COOLDOWN_FRAMES}
detections       = {'P1': [], 'P2': []}
last_label       = {'P1': None, 'P2': None}
last_label_frame = {'P1': -999, 'P2': -999}
prev_skel        = {'P1': None, 'P2': None}
racket_near      = {'P1': False, 'P2': False}

frame_idx  = 0
ball_detections_total   = 0
racket_detections_total = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    canvas = frame.copy()


    yolo_results = yolo(frame, classes=[CLS_PERSON, CLS_BALL, CLS_RACKET],
                        verbose=False)[0]

    person_boxes = []
    ball_boxes   = []
    racket_boxes = []

    for box in yolo_results.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        x1,y1,x2,y2 = map(int, box.xyxy[0])

        if cls_id == CLS_PERSON and conf > 0.4:
            person_boxes.append((x1,y1,x2,y2))
        elif cls_id == CLS_BALL and conf > 0.3:
            ball_boxes.append((x1,y1,x2,y2, conf))
            ball_detections_total += 1
        elif cls_id == CLS_RACKET and conf > 0.3:
            racket_boxes.append((x1,y1,x2,y2, conf))
            racket_detections_total += 1


    min_height = height * 0.15
    person_boxes = [b for b in person_boxes if (b[3]-b[1]) >= min_height]


    person_boxes.sort(key=lambda b: (b[2]-b[0])*(b[3]-b[1]), reverse=True)
    main_boxes = person_boxes[:2]
    main_boxes.sort(key=lambda b: (b[0]+b[2])//2)

    player_boxes = {}
    if len(main_boxes) >= 1: player_boxes['P1'] = main_boxes[0]
    if len(main_boxes) >= 2: player_boxes['P2'] = main_boxes[1]


    ball_pos = None
    if ball_boxes:

        ball_boxes.sort(key=lambda b: b[4], reverse=True)
        bx1,by1,bx2,by2,_ = ball_boxes[0]
        ball_pos = ((bx1+bx2)//2, (by1+by2)//2)


    racket_centers = []
    for rx1,ry1,rx2,ry2,_ in racket_boxes:
        racket_centers.append(((rx1+rx2)//2, (ry1+ry2)//2))


    player_skels = {}
    for pid, pose_det in [('P1', pose_p1), ('P2', pose_p2)]:
        if pid in player_boxes:
            pts = extract_skeleton_from_crop(pose_det, frame, player_boxes[pid])
            if pts is None:
                pts = prev_skel[pid] if prev_skel[pid] is not None \
                      else [[0.0,0.0]] * len(SELECTED_JOINTS)
            prev_skel[pid] = pts
        else:
            pts = prev_skel[pid] if prev_skel[pid] is not None \
                  else [[0.0,0.0]] * len(SELECTED_JOINTS)
        player_skels[pid] = pts


    def ball_near_player_yolo(ball_pos, player_box):
        """Ball within player bounding box expanded by 20% to avoid tracking the third far player which happen in the first run, that is prove of more that two players detected at the same time"""
        if ball_pos is None or player_box is None:
            return False
        x1,y1,x2,y2 = player_box
        pad_x = int((x2-x1) * 0.2)
        pad_y = int((y2-y1) * 0.2)
        return (x1-pad_x <= ball_pos[0] <= x2+pad_x and
                y1-pad_y <= ball_pos[1] <= y2+pad_y)


    def racket_near_player(racket_centers, player_box):
        """Any racket center =box"""
        if not racket_centers or player_box is None:
            return False
        x1,y1,x2,y2 = player_box
        for rx,ry in racket_centers:
            if x1 <= rx <= x2 and y1 <= ry <= y2:
                return True
        return False

    ball_player = None
    for pid in ['P1','P2']:
        if pid in player_boxes:
            if ball_near_player_yolo(ball_pos, player_boxes[pid]):
                ball_player = pid
                break

    for pid in ['P1','P2']:
        racket_near[pid] = racket_near_player(
            racket_centers,
            player_boxes.get(pid, None)
        )


    for pid in ['P1', 'P2']:
        pts = player_skels[pid]
        buffers[pid].append(np.array(pts).flatten())
        if len(buffers[pid]) > WINDOW_SIZE:
            buffers[pid].pop(0)

        if (len(buffers[pid]) == WINDOW_SIZE
                and (frame_idx % STRIDE == 0)
                and (frame_idx - last_det[pid]) >= COOLDOWN_FRAMES):

            cls, conf = hip_normalise_and_predict(buffers[pid])

            if conf >= THRESHOLD:

                pbox = player_boxes.get(pid, None)
                ball_ok   = ball_near_player_yolo(ball_pos, pbox)
                racket_ok = racket_near[pid]

                if ball_ok or racket_ok:
                    gate = "ball+racket" if (ball_ok and racket_ok) \
                           else ("ball" if ball_ok else "racket")
                    t_sec = frame_idx / fps
                    detections[pid].append((frame_idx, cls, conf, t_sec))
                    last_det[pid]         = frame_idx
                    last_label[pid]       = (cls, conf)
                    last_label_frame[pid] = frame_idx
                    print(f"  {pid}  frame={frame_idx:5d}  t={t_sec:.2f}s  "
                          f"{LABEL_MAP[cls]:<10}  conf={conf:.3f}  [{gate}]")




    for pid, col in PLAYER_COLORS.items():
        if pid in player_boxes:
            x1,y1,x2,y2 = player_boxes[pid]
            cv2.rectangle(canvas, (x1,y1), (x2,y2), col, 2)
            label_txt = pid
            if racket_near[pid]: label_txt += " 🎾"
            cv2.putText(canvas, label_txt, (x1, y1-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)


    for pid in ['P1','P2']:
        col = PLAYER_COLORS[pid]
        pts = player_skels[pid]
        for j in range(len(pts)):
            x = int(pts[j][0] * width)
            y = int(pts[j][1] * height)
            cv2.circle(canvas, (x,y), 5, col, -1)


    if ball_pos:
        bx, by = ball_pos
        ball_col = PLAYER_COLORS.get(ball_player, (0,255,255)) \
                   if ball_player else (0,255,255)
        cv2.circle(canvas, (bx,by), 14, ball_col, 3)
        cv2.putText(canvas, "BALL", (bx+16,by),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, ball_col, 2)

    for rx1,ry1,rx2,ry2,rconf in racket_boxes:
        cv2.rectangle(canvas, (rx1,ry1), (rx2,ry2), (0,200,255), 2)
        cv2.putText(canvas, f"RACKET {rconf:.2f}", (rx1, ry1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,200,255), 1)


    for pid in ['P1','P2']:
        show_label = (last_label[pid] is not None and
                      last_label_frame[pid] <= frame_idx and
                      frame_idx < last_label_frame[pid] + WINDOW_SIZE)
        if show_label:
            cls, conf = last_label[pid]
            label = LABEL_MAP[cls]
            col   = STROKE_COLORS[cls]
            x_pos = 30 if pid == 'P1' else width//2
            y_pos = 120 if pid == 'P1' else 220
            cv2.putText(canvas, f"{pid}: {label.upper()}",
                        (x_pos, y_pos),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0,0,0), 8)
            cv2.putText(canvas, f"{pid}: {label.upper()}",
                        (x_pos, y_pos),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.8, col, 4)
            cv2.putText(canvas, f"conf={conf:.2f}",
                        (x_pos, y_pos+40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)

    for i, pid in enumerate(['P1','P2']):
        dets = detections[pid]
        f_c = sum(1 for d in dets if d[1]==0)
        b_c = sum(1 for d in dets if d[1]==1)
        s_c = sum(1 for d in dets if d[1]==2)
        col = PLAYER_COLORS[pid]
        cv2.putText(canvas, f"{pid}  F:{f_c} B:{b_c} S:{s_c}",
                    (width-320, 40 + i*40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,0), 4)
        cv2.putText(canvas, f"{pid}  F:{f_c} B:{b_c} S:{s_c}",
                    (width-320, 40 + i*40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, col, 2)

    cv2.putText(canvas, f"t={frame_idx/fps:.2f}s",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255,255,255), 2)

    out.write(canvas)
    frame_idx += 1

    if frame_idx % 150 == 0:
        pct = frame_idx/total*100 if total > 0 else 0
        print(f"  Progress: {frame_idx}/{total} ({pct:.0f}%)  "
              f"Ball:{ball_detections_total}  Racket:{racket_detections_total}  "
              f"P1:{len(detections['P1'])}det  P2:{len(detections['P2'])}det")

cap.release()
out.release()
pose_p1.close()
pose_p2.close()

print("-" * 55)
print(f"Done: {frame_idx} frames")
print(f"Ball detections  : {ball_detections_total} frames")
print(f"Racket detections: {racket_detections_total} frames")
print(f"P1 strokes: {len(detections['P1'])}  |  P2 strokes: {len(detections['P2'])}")
print(f"Output: {output_name}")

 Processing: 60fps  848x478  833 frames
   Output -> 2Player (1)_multi_human.mp4
-------------------------------------------------------
  P2  frame=   70  t=1.17s  Forehand    conf=1.000  [racket]
  P1  frame=   90  t=1.50s  Backhand    conf=1.000  [ball]
  Progress: 150/833 (18%)  Ball:4  Racket:42  P1:1det  P2:1det
  P2  frame=  265  t=4.42s  Forehand    conf=1.000  [racket]
  Progress: 300/833 (36%)  Ball:4  Racket:87  P1:1det  P2:2det
  Progress: 450/833 (54%)  Ball:4  Racket:106  P1:1det  P2:2det
  Progress: 600/833 (72%)  Ball:4  Racket:135  P1:1det  P2:2det
  P1  frame=  720  t=12.01s  Backhand    conf=1.000  [racket]
  Progress: 750/833 (90%)  Ball:4  Racket:155  P1:2det  P2:2det
-------------------------------------------------------
Done: 833 frames
Ball detections  : 4 frames
Racket detections: 161 frames
P1 strokes: 2  |  P2 strokes: 2
Output: 2Player (1)_multi_human.mp4


In [15]:



ball_found = ball_found if 'ball_found' in vars() else 0
frame_idx  = frame_idx  if 'frame_idx'  in vars() else 0

print("\n" + "="*55)
print("MULTI-HUMAN DETECTION RESULTS")
print("="*55)

for pid in ['P1','P2']:
    dets = detections[pid]
    f_c  = sum(1 for d in dets if d[1]==0)
    b_c  = sum(1 for d in dets if d[1]==1)
    s_c  = sum(1 for d in dets if d[1]==2)
    print(f"\n{pid} — {len(dets)} strokes detected")
    print(f"  Forehand : {f_c}")
    print(f"  Backhand : {b_c}")
    print(f"  Serve    : {s_c}")
    if dets:
        print(f"  Times    : {[round(d[3],2) for d in dets]}")

print(f"\nTotal players tracked : {len([p for p in ['P1','P2'] if detections[p]])}")
print(f"Ball detections       : {ball_found}/{frame_idx} frames")
print("="*55)


stats = np.zeros((height, width, 3), dtype=np.uint8)
cv2.putText(stats, "MULTI-HUMAN STROKE DETECTION",
            (max(0, width//2-380), 70),
            cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,255,255), 3)
cv2.putText(stats, "YOLOv8 + MediaPipe + CNN-LSTM",
            (max(0, width//2-320), 120),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (180,180,180), 2)

y_pos = 200
for pid in ['P1','P2']:
    dets = detections[pid]
    f_c  = sum(1 for d in dets if d[1]==0)
    b_c  = sum(1 for d in dets if d[1]==1)
    s_c  = sum(1 for d in dets if d[1]==2)
    col  = PLAYER_COLORS[pid]
    cv2.putText(stats,
                f"{pid} — {len(dets)} strokes  (F:{f_c}  B:{b_c}  S:{s_c})",
                (max(0, width//2-350), y_pos),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, col, 3)
    y_pos += 80


final_name = output_name.replace('.mp4','') + '_final.mp4'
out2 = cv2.VideoWriter(final_name,
                       cv2.VideoWriter_fourcc(*'mp4v'),
                       fps, (width, height))


cap3 = cv2.VideoCapture(output_name)
frame_count = 0
while cap3.isOpened():
    ret, f = cap3.read()
    if not ret: break
    out2.write(f)
    frame_count += 1
cap3.release()
print(f"Copied {frame_count} frames from annotated video")


stats_frames = int(fps * 4)
for _ in range(stats_frames):
    out2.write(stats)

out2.release()
print(f"\nFinal video: {final_name}")
print(f"Total frames: {frame_count} + {stats_frames} stats = {frame_count + stats_frames}")


files.download(final_name)
print(f"Downloading {final_name}")


MULTI-HUMAN DETECTION RESULTS

P1 — 2 strokes detected
  Forehand : 0
  Backhand : 2
  Serve    : 0
  Times    : [1.5, 12.01]

P2 — 2 strokes detected
  Forehand : 2
  Backhand : 0
  Serve    : 0
  Times    : [1.17, 4.42]

Total players tracked : 2
Ball detections       : 0/833 frames
Copied 833 frames from annotated video

Final video: 2Player (1)_multi_human_final.mp4
Total frames: 833 + 239 stats = 1072


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>